In [2]:
import mon2y
import optuna
import pandas as pd
import numpy as np
from IPython.display import display

In [3]:
target_game_turns = 100

study = optuna.create_study(
    storage="sqlite:///db.sqlite3",  # Specify the storage URL here.
    study_name="connect4_100_turns"
)

[I 2025-12-23 14:31:24,463] A new study created in RDB with name: connect4_100_turns


In [ ]:
def objective(trial: optuna.):
    board_width = trial.suggest_int("board_width", 0, 1000)
    board_height = trial.suggest_int("board_height", 0, 1000)

    trial.

    raw_results = mon2y.explore(mon2y.Games.C4, 100, 4, hyperparams={"board_width":board_width,"board_height":board_height})
    df = pd.DataFrame(raw_results)
    df['ratio']= (df['turns']-df['rwalk'])/df['turns']
    df['norm_sum_diff_est_reward'] = (df['sum_diff_est_reward'] - df['sum_diff_est_reward'].min()) / (df['sum_diff_est_reward'].max() - df['sum_diff_est_reward'].min())
    df['trust'] = df['ratio'] * df['norm_sum_diff_est_reward']
    df['norm_trust'] = (df['trust']-df['trust'].min()) / (df['trust'].max() - df['trust'].min())
    T_target = 21
    
    w = df['norm_trust']
    Neff = w.sum()
    
    # ---- Turns ----
    mu_T = (w * df['turns']).sum() / Neff
    var_T = (w * (df['turns'] - mu_T)**2).sum() / Neff
    se_T = np.sqrt(var_T / Neff)
    z_T = (mu_T - target_game_turns) / se_T
    
    # ---- Win-rate ----
    wins = (df['winning_player'] == 1).astype(float)
    p_hat = (w * wins).sum() / Neff
    se_p = np.sqrt(p_hat * (1 - p_hat) / Neff)
    z_p = (p_hat - 0.5) / se_p
    
    score = abs(z_T) + abs(z_p)

    return score

In [ ]:
study.optimize(objective, n_trials=100)
print(f"Best value: {study.best_value} (params: {study.best_params})")
